# Zero-shot Jev on the validation reviews (baseline for the trained models)

How far is a fine-tuned decision model from zero-shot Jev? The Jev numbers in FINDINGS.md are on the CLEF TAR
**test** reviews, so they cannot be compared with the trained models' **val** numbers. This notebook runs Jev on
exactly the val requests the trained model is evaluated on.

| | |
|---|---|
| Requests | `data/processed/training/requests/<config>/val.jsonl` from [scripts/build_requests.py](../../scripts/build_requests.py): review = selection criteria, record = title + abstract, one `noul` question `include` |
| What Jev gets | the same request as the trained model; only the label is removed and the pinned model is added. Zero-shot: no labels are sent |
| Model | `typesafe/jev-1.14` (responses report `typesafe/jev-1.14-20261014`) |
| Reviews | 17 CLEF TAR 2019 Intervention training reviews (val), 27,870 records |
| Labels | CLEF TAR **abstract-level** (17 reviews) and **content-level** (`final`, 16 reviews) |
| Ranking | P(include) descending; ties (Jev returns two decimals) broken by BM25 with the review title as query, then record id. Trained models: ties by record id, as in the training notebook |
| Cache | raw responses in `results/requests__<config>__val/typesafe_jev-1.14/<review>.jsonl`; metrics only from the cache. A failed call is ranked last and counted, never dropped |

Calling code: [scripts/jev_on_requests.py](../../scripts/jev_on_requests.py) (also runs from the command line).

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "scripts"))
import screening as S
import jev_on_requests as J

REQUESTS = "review-criteria__record-title+abstract__label-abstract_else_final"
SPLIT = "val"
RUN_PAID = False   # True: call the API for records not yet in the cache (about $1.60 for all of val)

req = J.DATA / "requests" / REQUESTS
rows, meta = J.read_jsonl(req / f"{SPLIT}.jsonl"), J.read_jsonl(req / f"{SPLIT}.meta.jsonl")
reviews = sorted({m["review_id"] for m in meta})
OUT = S.out_dir(f"requests__{REQUESTS}__{SPLIT}")
print(len(rows), "requests,", len(reviews), "reviews | cache:", OUT.relative_to(ROOT))
print(json.dumps(J.body_of(rows[0]), indent=1, ensure_ascii=False)[:1200])

## Cost estimate (no API call)

In [ ]:
chars = np.array([len(json.dumps(J.body_of(r))) for r in rows])
price = S.price_per_input_token()
print(f"mean chars {chars.mean():.0f} | est. input tokens {chars.sum() / 3.06 / 1e6:.1f}M | est. cost ${chars.sum() / 3.06 * price:.2f}")

## API calls

Only records without a cached response are called. Try one small review first (e.g. `CD010355`, 43 records).

In [ ]:
if RUN_PAID:
    by_review = {}
    for row, m in zip(rows, meta):
        by_review.setdefault(m["review_id"], []).append((m["record_id"], J.body_of(row)))
    for review in reviews:
        J.run_review(review, by_review[review], OUT, workers=16)

## Jev metrics from the cache

In [ ]:
jev = J.report(meta, OUT, reviews)

## Comparison on the same records

Random order (expected value over 50 shuffles), BM25 (query = review title), zero-shot Jev, and every training run
in `runs/` that has val scores: its step 0 (untrained) and its best step by abstract-level MAP.
**Choosing the best step on val makes the trained numbers slightly optimistic**; the test split decides later.

In [ ]:
def per_review(df, col, level, tiebreak=("record_id",)):
    out = {}
    for rid, g in df.groupby("review_id"):
        y = g[f"label_{level}"].fillna(0).astype(int)
        if y.sum():
            order = g.assign(y=y).sort_values([col, *tiebreak], ascending=[False, *[t == "record_id" for t in tiebreak]])
            out[rid] = S.metrics(order["y"])
    return out

base = jev[["review_id", "record_id", "jev", "bm25", "label_abstract", "label_final"]].copy()
scores = {"BM25": (base, "bm25", ("record_id",)), "Jev zero-shot": (base, "jev", ("bm25", "record_id"))}
for run in sorted((ROOT / "runs").glob("*/val_metrics.jsonl")):
    hist = pd.read_json(run, lines=True).drop_duplicates("step", keep="last").set_index("step")
    name = run.parent.name.replace("__review-criteria__record-title+abstract__label-abstract_else_final", "")
    for label, step in (("step 0", 0), ("best", hist.map_abstract.idxmax())):
        f = run.parent / "val_scores" / f"step-{step:06d}.jsonl"
        if f.exists() and (label == "step 0" or step != 0):
            s = pd.read_json(f, lines=True)[["review_id", "record_id", "p"]]
            scores[f"{name}, {label} ({step})"] = (base.merge(s, on=["review_id", "record_id"]), "p", ("record_id",))

rng = np.random.default_rng(0)
table, per = {}, {}
for level in ("abstract", "final"):
    rand = {}
    for rid, g in base.groupby("review_id"):
        y = g[f"label_{level}"].fillna(0).astype(int).values
        if y.sum():
            rand[rid] = pd.DataFrame([S.metrics(rng.permutation(y)) for _ in range(50)]).mean().to_dict()
    per[level, "Random order"] = rand
    for name, (df, col, tb) in scores.items():
        per[level, name] = per_review(df, col, level, tb)
    for (lvl, name), d in per.items():
        if lvl == level:
            m = pd.DataFrame(d.values()).mean() * 100
            table[name, level] = {k: round(m[k], 1) for k in ("MAP", "R@10%", "WSS@95%", "TNR@95%")} | {"reviews": len(d)}
table = pd.DataFrame(table).T
table.index.names = ["", "labels"]
table.sort_index(level=1, sort_remaining=False)

## Per review (abstract-level labels, MAP)

In [ ]:
counts = base.groupby("review_id").agg(records=("record_id", "size"), includes=("label_abstract", "sum"))
names = [n for (lvl, n) in per if lvl == "abstract"]
by_review = pd.DataFrame({n: {r: round(100 * m["MAP"], 1) for r, m in per["abstract", n].items()} for n in names})
by_review = counts.join(by_review, how="right")
by_review.sort_values("Jev zero-shot", ascending=False)

Difference to Jev per review: mean with 95% bootstrap CI over reviews, and how many reviews are better / worse.

In [ ]:
def compare(level, a, b="Jev zero-shot", metric="MAP"):
    common = sorted(set(per[level, a]) & set(per[level, b]))
    d = np.array([100 * (per[level, a][r][metric] - per[level, b][r][metric]) for r in common])
    bs = [np.random.default_rng(i).choice(d, len(d)).mean() for i in range(5000)]
    return {"mean": round(d.mean(), 1), "CI low": round(np.percentile(bs, 2.5), 1), "CI high": round(np.percentile(bs, 97.5), 1),
            "better": int((d > 0).sum()), "worse": int((d < 0).sum()), "reviews": len(d)}

pd.DataFrame({(n, lvl, m): compare(lvl, n, metric=m) for (lvl, n) in per if n not in ("Jev zero-shot", "Random order")
              for m in ("MAP", "WSS@95%")}).T